[![](imagens/colab-badge.png){width="16%"}](https://colab.research.google.com/github/fzampirolli/pdi-vc/blob/master/notebooks_alunos/cap03/cap03.EPs_aluno.ipynb)
[![](imagens/github-badge.png){width="20%"}](https://github.com/fzampirolli/pdi-vc)

## 💻 **Parte Prática com Exercícios de Programação**

---

### 🎯 Objetivo deste Caderno {.unnumbered}

O caderno permite desenvolver, validar, organizar e testar soluções de **Exercícios de Programação (EPs)** em ambientes interativos, como o Colab, com os mesmos casos de teste do Moodle, copiando para lá apenas na hora de registrar a nota oficial.

#### Download {.unnumbered}

Baixe `morph.py` e `testsuite.py` executando a célula abaixo:

In [1]:
import os, sys, importlib, inspect, urllib.request

BASE_URL = "https://raw.githubusercontent.com/fzampirolli/pdi-vc/master/morph"
for f in ["morph.py", "testsuite.py"]:
    if not os.path.exists(f):
        urllib.request.urlretrieve(f"{BASE_URL}/{f}", f)

import morph, testsuite
importlib.reload(morph); importlib.reload(testsuite)
from morph import mm
from testsuite import TestSuite

print(f"\u2705 Ambiente pronto. Morph: {morph.__version__} | TestSuite: {testsuite.__version__}")

✅ Ambiente pronto. Morph: 1.1.0 | TestSuite: 1.1.0


---

#### Executando os Testes {.unnumbered}

Para rodar os testes, execute `TestSuite("EP03_XX.extensão").run()` numa nova célula, trocando a extensão pela da linguagem usada (`.py`, `.java`, `.c`, `.cpp`, `.js` ou `.r`). O sistema baixa os casos de teste do GitHub, executa o programa e calcula a nota automaticamente.

---

### EP03_01 ➕ Adição Saturada de Constante

Em sistemas de vigilância por vídeo, câmeras em ambientes com iluminação variável produzem imagens subexpostas. O ajuste de brilho por **adição saturada de uma constante** é a operação mais simples para correção imediata, sendo aplicada em tempo real nos chips de câmeras embarcadas e em pipelines de pré-processamento de robôs móveis.

#### 📋 Diretrizes de Implementação

1. **Dimensões:** Ler os inteiros $L$ (linhas) e $C$ (colunas).
2. **Constante:** Ler o inteiro $k$ (valor a ser somado).
3. **Dados:** Ler os valores inteiros da matriz original pixel a pixel.
4. **Mapeamento:** Para cada pixel $p$, calcular o novo valor pela equação:

$$p' = \text{clip}(p + k)$$

5. **Saída:** Exibir a matriz resultante com dimensões $L \times C$.

---

#### 📌 Restrições Computacionais

* **Saturação (*Clipping*):** Os valores devem ser confinados ao intervalo $[0, 255]$:
$$\text{clip}(x) = \max(0, \min(255, x))$$
* **Tipo:** O resultado final deve ser inteiro (sem casas decimais).
* **$k$ pode ser negativo:** valores negativos escurecem a imagem; positivos clareiam.

---

#### 🧠 Fundamentação Teórica

| Parâmetro | Tipo | Impacto Visual |
|-----------|------|----------------|
| **$k > 0$** | Inteiro | Clareia a imagem; pixels próximos de 255 saturam em branco |
| **$k < 0$** | Inteiro | Escurece a imagem; pixels próximos de 0 saturam em preto |
| **$k = 0$** | Inteiro | Imagem inalterada |

---

#### 📦 Especificação de Entrada e Saída (VPL)

**Entrada:**
* Linha 1: Inteiro $L$.
* Linha 2: Inteiro $C$.
* Linha 3: Inteiro $k$.
* Linhas seguintes: Elementos inteiros da matriz original.

**Saída:**
* Matriz transformada em $L$ linhas e $C$ colunas, valores inteiros separados por espaço.

#### 📌 Exemplos

| Entrada | Saída | Observação |
|---------|-------|------------|
| 2<br>3<br>50<br>0 100 200<br>210 240 255 | 50 150 250<br>255 255 255 | Saturação em 255 nos pixels altos |
| 1<br>4<br>-30<br>0 20 200 255 | 0 0 170 225 | Saturação em 0 nos pixels baixos |

In [2]:
from IPython.display import HTML
HTML("""
<div id="sim-ep0301" style="background-color:#fef9ef;border-radius:18px;border:1px solid #ede6d8;overflow:hidden;margin-top:20px;font-family:sans-serif;">
  <div style="background:#f3efe6;padding:8px 16px;font-size:12px;color:#5e5a4a;border-bottom:1px solid #e9dfcf;display:flex;justify-content:space-between;align-items:center;">
    <span>🎮 Simulador: Adição Saturada de Constante</span>
    <span style="background:#e8e0cf;border-radius:40px;padding:2px 10px;font-weight:600;font-size:10px;">➕ p' = clip(p + k)</span>
  </div>
  <div style="padding:20px;background:white;">
    <div style="background:#fafafa;border:1px solid #ddd;border-radius:12px;padding:20px;margin-bottom:20px;">
      <div style="display:flex;justify-content:space-between;margin-bottom:8px;">
        <label style="font-size:12px;font-weight:bold;color:#27ae60;">k (Constante)</label>
        <span id="ep0301_vl_k" style="font-family:monospace;font-weight:bold;color:#27ae60;">0</span>
      </div>
      <input id="ep0301_sl_k" style="width:100%;accent-color:#27ae60;" max="128" min="-128" step="1" type="range" value="0">
    </div>
    <div style="display:grid;grid-template-columns:1fr 1fr;gap:20px;">
      <div style="text-align:center;background:white;border:1px solid #eee;padding:15px;border-radius:12px;">
        <p style="font-size:10px;font-weight:600;color:#888;text-transform:uppercase;margin-bottom:12px;">Entrada Original</p>
        <div id="ep0301_grid_orig" style="display:grid;grid-template-columns:repeat(4,42px);gap:4px;justify-content:center;"></div>
        <button id="ep0301_btn_new" style="margin-top:15px;font-size:11px;padding:5px 10px;border-radius:4px;border:1px solid #ccc;background:white;cursor:pointer;">🎲 Nova Imagem</button>
      </div>
      <div style="text-align:center;background:white;border:1px solid #eee;padding:15px;border-radius:12px;">
        <p style="font-size:10px;font-weight:600;color:#888;text-transform:uppercase;margin-bottom:12px;">Resultado (p')</p>
        <div id="ep0301_grid_new" style="display:grid;grid-template-columns:repeat(4,42px);gap:4px;justify-content:center;"></div>
        <button id="ep0301_btn_reset" style="margin-top:15px;font-size:11px;padding:5px 10px;border-radius:4px;border:1px solid #ccc;background:white;cursor:pointer;">↩ Resetar</button>
      </div>
    </div>
    <div id="ep0301_debug" style="margin-top:20px;background:#e8f5e9;border-radius:8px;padding:10px;border:1px solid #c8e6c9;font-family:monospace;font-size:11px;color:#2e7d32;text-align:center;">Fórmula: clip(p + 0)</div>
  </div>
</div>
<script>
setTimeout(function(){
  var slK=document.getElementById('ep0301_sl_k');
  var vlK=document.getElementById('ep0301_vl_k');
  var gO=document.getElementById('ep0301_grid_orig');
  var gN=document.getElementById('ep0301_grid_new');
  var db=document.getElementById('ep0301_debug');
  if(!slK||!gO)return;
  var pixels=[];
  function generate_0301(){pixels=Array.from({length:16},function(){return Math.floor(Math.random()*256);});}
  function render_0301(){
    var k=parseInt(slK.value);
    vlK.innerText=k;
    db.innerHTML='Fórmula aplicada: <b>clip(p + ('+k+'))</b>';
    gO.innerHTML=''; gN.innerHTML='';
    pixels.forEach(function(p){
      var cO=document.createElement('div');
      cO.style.cssText='width:42px;height:42px;display:flex;align-items:center;justify-content:center;font-size:10px;font-weight:bold;border-radius:4px;border:1px solid #eee;';
      cO.style.background='rgb('+p+','+p+','+p+')';
      cO.style.color=p>128?'black':'white';
      cO.innerText=p; gO.appendChild(cO);
      var res=Math.max(0,Math.min(255,p+k));
      var cN=document.createElement('div');
      cN.style.cssText='width:42px;height:42px;display:flex;align-items:center;justify-content:center;font-size:10px;font-weight:bold;border-radius:4px;border:1px solid #ddd;';
      cN.style.background='rgb('+res+','+res+','+res+')';
      cN.style.color=res>128?'black':'white';
      cN.innerText=res; gN.appendChild(cN);
    });
  }
  slK.oninput=render_0301;
  document.getElementById('ep0301_btn_new').onclick=function(){generate_0301();render_0301();};
  document.getElementById('ep0301_btn_reset').onclick=function(){slK.value=0;render_0301();};
  generate_0301(); render_0301();
},100);
</script>
""")

In [ ]:
%%writefile EP03_01.py
# Código Python

In [ ]:
TestSuite("EP03_01.py").run()

---

### EP03_02 🔀 Alpha Blending de Duas Imagens

Em medicina nuclear, imagens de diferentes modalidades (tomografia computadorizada e ressonância magnética) são fundidas para auxiliar no diagnóstico. A **mistura ponderada** (*alpha blending*) é a operação fundamental desse processo, permitindo ao radiologista controlar interativamente o peso de cada modalidade na imagem exibida.

#### 📋 Diretrizes de Implementação

1. **Dimensões:** Ler os inteiros $L$ (linhas) e $C$ (colunas).
2. **Parâmetro:** Ler o valor real $\alpha \in [0, 1]$.
3. **Dados:** Ler os valores inteiros da matriz $f_1$ (imagem 1) e em seguida da matriz $f_2$ (imagem 2).
4. **Mapeamento:** Para cada posição $(i, j)$, calcular:

$$g(i,j) = \text{clip}\left(\text{round}\left(\alpha \cdot f_1(i,j) + (1-\alpha) \cdot f_2(i,j)\right)\right)$$

5. **Saída:** Exibir a matriz resultante $L \times C$.

---

#### 📌 Restrições Computacionais

* **Arredondamento:** Aplicar `round` antes da conversão para inteiro.
* **Saturação:** Confinar ao intervalo $[0, 255]$ com $\text{clip}(x) = \max(0, \min(255, x))$.
* **Operação em float:** Realize a operação em ponto flutuante antes de arredondar.

---

#### 🧠 Fundamentação Teórica

| Valor de $\alpha$ | Resultado |
|:-----------------:|:----------|
| $\alpha = 1.0$ | Apenas $f_1$ |
| $\alpha = 0.5$ | Média aritmética de $f_1$ e $f_2$ |
| $\alpha = 0.0$ | Apenas $f_2$ |

---

#### 📦 Especificação de Entrada e Saída (VPL)

**Entrada:**
* Linha 1: Inteiro $L$.
* Linha 2: Inteiro $C$.
* Linha 3: Real $\alpha$.
* Linhas seguintes: Elementos de $f_1$ ($L$ linhas com $C$ valores cada).
* Linhas seguintes: Elementos de $f_2$ ($L$ linhas com $C$ valores cada).

**Saída:**
* Matriz resultante $L \times C$.

#### 📌 Exemplos

| Entrada | Saída | Observação |
|---------|-------|------------|
| 1<br>3<br>0.5<br>0 100 200<br>100 200 50 | 50 150 125 | Média entre as duas imagens |
| 1<br>3<br>1.0<br>10 20 30<br>90 80 70 | 10 20 30 | Apenas $f_1$ (alpha=1) |

In [3]:
from IPython.display import HTML
HTML("""
<div id="sim-ep0302" style="background-color:#fef9ef;border-radius:18px;border:1px solid #ede6d8;overflow:hidden;margin-top:20px;font-family:sans-serif;">
  <div style="background:#f3efe6;padding:8px 16px;font-size:12px;color:#5e5a4a;border-bottom:1px solid #e9dfcf;display:flex;justify-content:space-between;align-items:center;">
    <span>🎮 Simulador: Alpha Blending</span>
    <span style="background:#e8e0cf;border-radius:40px;padding:2px 10px;font-weight:600;font-size:10px;">🔀 g = α·f1 + (1-α)·f2</span>
  </div>
  <div style="padding:20px;background:white;">
    <div style="background:#fafafa;border:1px solid #ddd;border-radius:12px;padding:20px;margin-bottom:20px;">
      <div style="display:flex;justify-content:space-between;margin-bottom:8px;">
        <label style="font-size:12px;font-weight:bold;color:#8e44ad;">α (Alpha)</label>
        <span id="ep0302_vl_a" style="font-family:monospace;font-weight:bold;color:#8e44ad;">0.5</span>
      </div>
      <input id="ep0302_sl_a" style="width:100%;accent-color:#8e44ad;" max="1" min="0" step="0.05" type="range" value="0.5">
    </div>
    <div style="display:grid;grid-template-columns:1fr 1fr 1fr;gap:12px;">
      <div style="text-align:center;background:white;border:1px solid #eee;padding:12px;border-radius:12px;">
        <p style="font-size:10px;font-weight:600;color:#888;text-transform:uppercase;margin-bottom:8px;">Imagem f1</p>
        <div id="ep0302_grid_f1" style="display:grid;grid-template-columns:repeat(4,38px);gap:3px;justify-content:center;"></div>
        <button id="ep0302_btn_new" style="margin-top:10px;font-size:11px;padding:4px 8px;border-radius:4px;border:1px solid #ccc;background:white;cursor:pointer;">🎲 Novas</button>
      </div>
      <div style="text-align:center;background:white;border:1px solid #eee;padding:12px;border-radius:12px;">
        <p style="font-size:10px;font-weight:600;color:#888;text-transform:uppercase;margin-bottom:8px;">Imagem f2</p>
        <div id="ep0302_grid_f2" style="display:grid;grid-template-columns:repeat(4,38px);gap:3px;justify-content:center;"></div>
      </div>
      <div style="text-align:center;background:white;border:1px solid #eee;padding:12px;border-radius:12px;">
        <p style="font-size:10px;font-weight:600;color:#888;text-transform:uppercase;margin-bottom:8px;">Resultado g</p>
        <div id="ep0302_grid_g" style="display:grid;grid-template-columns:repeat(4,38px);gap:3px;justify-content:center;"></div>
        <button id="ep0302_btn_reset" style="margin-top:10px;font-size:11px;padding:4px 8px;border-radius:4px;border:1px solid #ccc;background:white;cursor:pointer;">↩ Reset</button>
      </div>
    </div>
    <div id="ep0302_debug" style="margin-top:16px;background:#f3e5f5;border-radius:8px;padding:10px;border:1px solid #ce93d8;font-family:monospace;font-size:11px;color:#6a1b9a;text-align:center;">Fórmula: clip(round(0.5 * f1 + 0.5 * f2))</div>
  </div>
</div>
<script>
setTimeout(function(){
  var slA=document.getElementById('ep0302_sl_a');
  var vlA=document.getElementById('ep0302_vl_a');
  var gF1=document.getElementById('ep0302_grid_f1');
  var gF2=document.getElementById('ep0302_grid_f2');
  var gG=document.getElementById('ep0302_grid_g');
  var db=document.getElementById('ep0302_debug');
  if(!slA||!gF1)return;
  var px1=[],px2=[];
  function gen_0302(){px1=Array.from({length:16},function(){return Math.floor(Math.random()*256);});px2=Array.from({length:16},function(){return Math.floor(Math.random()*256);});}
  function cell_0302(v){
    var c=document.createElement('div');
    c.style.cssText='width:38px;height:38px;display:flex;align-items:center;justify-content:center;font-size:9px;font-weight:bold;border-radius:4px;border:1px solid #eee;';
    c.style.background='rgb('+v+','+v+','+v+')';
    c.style.color=v>128?'black':'white';
    c.innerText=v; return c;
  }
  function render_0302(){
    var a=parseFloat(slA.value);
    vlA.innerText=a.toFixed(2);
    db.innerHTML='Fórmula: <b>clip(round('+a.toFixed(2)+' * f1 + '+( 1-a).toFixed(2)+' * f2))</b>';
    gF1.innerHTML=''; gF2.innerHTML=''; gG.innerHTML='';
    for(var i=0;i<16;i++){
      gF1.appendChild(cell_0302(px1[i]));
      gF2.appendChild(cell_0302(px2[i]));
      var r=Math.max(0,Math.min(255,Math.round(a*px1[i]+(1-a)*px2[i])));
      gG.appendChild(cell_0302(r));
    }
  }
  slA.oninput=render_0302;
  document.getElementById('ep0302_btn_new').onclick=function(){gen_0302();render_0302();};
  document.getElementById('ep0302_btn_reset').onclick=function(){slA.value=0.5;render_0302();};
  gen_0302(); render_0302();
},100);
</script>
""")

In [ ]:
%%writefile EP03_02.py
# Código Python

In [ ]:
TestSuite("EP03_02.py").run()

---

### EP03_03 🎭 Inversão de Imagem (Negativo Fotográfico)

Em radiologia, as imagens de raio-X são tradicionalmente visualizadas em negativo: ossos aparecem em preto sobre fundo branco. A operação de **negativo fotográfico** é aplicada rotineiramente em PACS (*Picture Archiving and Communication Systems*) para facilitar a detecção de fraturas e densidades ósseas.

#### 📋 Diretrizes de Implementação

1. **Dimensões:** Ler os inteiros $L$ (linhas) e $C$ (colunas).
2. **Dados:** Ler os valores inteiros da matriz original.
3. **Mapeamento:** Para cada pixel $p$, calcular o negativo:

$$p' = 255 - p$$

4. **Saída:** Exibir a matriz resultante $L \times C$.

---

#### 📌 Restrições Computacionais

* **Sem clipping necessário:** O resultado de $255 - p$ com $p \in [0, 255]$ é sempre $\in [0, 255]$.
* **Tipo inteiro:** Saída deve ser valores inteiros.
* **Equivalência lógica:** A operação é idêntica ao `NOT` bit a bit (`mm.bnot`) em imagens de 8 bits.

---

#### 🧠 Fundamentação Teórica

| Pixel Original $p$ | Pixel Negativo $p'$ | Observação |
|:------------------:|:-------------------:|:----------:|
| 0 (preto) | 255 (branco) | Inversão total |
| 128 (cinza médio) | 127 (cinza médio) | Valor central |
| 255 (branco) | 0 (preto) | Inversão total |

---

#### 📦 Especificação de Entrada e Saída (VPL)

**Entrada:**
* Linha 1: Inteiro $L$.
* Linha 2: Inteiro $C$.
* Linhas seguintes: Elementos inteiros da matriz original.

**Saída:**
* Matriz negativa em $L$ linhas e $C$ colunas.

#### 📌 Exemplos

| Entrada | Saída | Observação |
|---------|-------|------------|
| 1<br>4<br>0 128 200 255 | 255 127 55 0 | Inversão de cada pixel |
| 2<br>2<br>10 20<br>30 40 | 245 235<br>225 215 | Matriz 2x2 invertida |

In [4]:
from IPython.display import HTML
HTML("""
<div id="sim-ep0303" style="background-color:#fef9ef;border-radius:18px;border:1px solid #ede6d8;overflow:hidden;margin-top:20px;font-family:sans-serif;">
  <div style="background:#f3efe6;padding:8px 16px;font-size:12px;color:#5e5a4a;border-bottom:1px solid #e9dfcf;display:flex;justify-content:space-between;align-items:center;">
    <span>🎮 Simulador: Negativo Fotográfico</span>
    <span style="background:#e8e0cf;border-radius:40px;padding:2px 10px;font-weight:600;font-size:10px;">🎭 p' = 255 - p</span>
  </div>
  <div style="padding:20px;background:white;">
    <div style="display:grid;grid-template-columns:1fr 1fr;gap:20px;">
      <div style="text-align:center;background:white;border:1px solid #eee;padding:15px;border-radius:12px;">
        <p style="font-size:10px;font-weight:600;color:#888;text-transform:uppercase;margin-bottom:12px;">Entrada Original</p>
        <div id="ep0303_grid_orig" style="display:grid;grid-template-columns:repeat(4,42px);gap:4px;justify-content:center;"></div>
        <button id="ep0303_btn_new" style="margin-top:15px;font-size:11px;padding:5px 10px;border-radius:4px;border:1px solid #ccc;background:white;cursor:pointer;">🎲 Nova Imagem</button>
      </div>
      <div style="text-align:center;background:white;border:1px solid #eee;padding:15px;border-radius:12px;">
        <p style="font-size:10px;font-weight:600;color:#888;text-transform:uppercase;margin-bottom:12px;">Negativo (p'=255-p)</p>
        <div id="ep0303_grid_neg" style="display:grid;grid-template-columns:repeat(4,42px);gap:4px;justify-content:center;"></div>
      </div>
    </div>
    <div id="ep0303_debug" style="margin-top:20px;background:#fce4ec;border-radius:8px;padding:10px;border:1px solid #f48fb1;font-family:monospace;font-size:11px;color:#880e4f;text-align:center;">Fórmula: p' = 255 - p</div>
  </div>
</div>
<script>
setTimeout(function(){
  var gO=document.getElementById('ep0303_grid_orig');
  var gN=document.getElementById('ep0303_grid_neg');
  if(!gO)return;
  var pixels=[];
  function gen_0303(){pixels=Array.from({length:16},function(){return Math.floor(Math.random()*256);});}
  function render_0303(){
    gO.innerHTML=''; gN.innerHTML='';
    pixels.forEach(function(p){
      var cO=document.createElement('div');
      cO.style.cssText='width:42px;height:42px;display:flex;align-items:center;justify-content:center;font-size:10px;font-weight:bold;border-radius:4px;border:1px solid #eee;';
      cO.style.background='rgb('+p+','+p+','+p+')';
      cO.style.color=p>128?'black':'white';
      cO.innerText=p; gO.appendChild(cO);
      var r=255-p;
      var cN=document.createElement('div');
      cN.style.cssText='width:42px;height:42px;display:flex;align-items:center;justify-content:center;font-size:10px;font-weight:bold;border-radius:4px;border:1px solid #ddd;';
      cN.style.background='rgb('+r+','+r+','+r+')';
      cN.style.color=r>128?'black':'white';
      cN.innerText=r; gN.appendChild(cN);
    });
  }
  document.getElementById('ep0303_btn_new').onclick=function(){gen_0303();render_0303();};
  gen_0303(); render_0303();
},100);
</script>
""")

In [ ]:
%%writefile EP03_03.py
# Código Python

In [ ]:
TestSuite("EP03_03.py").run()

---

### EP03_04 📊 Equalização de Histograma (L bits)

Em imagens de satélite de sensoriamento remoto, a variação de iluminação ao longo do dia produz imagens de baixo contraste. A **equalização de histograma** é aplicada automaticamente em satélites como o Landsat para redistribuir os tons, revelando detalhes de vegetação, relevo e zonas urbanas invisíveis na imagem original.

#### 📋 Diretrizes de Implementação

1. **Dimensões:** Ler os inteiros $L$ (linhas), $C$ (colunas) e $B$ (número de bits, com $L_{\max} = 2^B$).
2. **Dados:** Ler a matriz de pixels $f$ com valores em $[0, 2^B - 1]$.
3. **Histograma:** Calcular $h[k]$ = número de pixels com intensidade $k$, para $k = 0 \ldots 2^B-1$.
4. **Probabilidade:** $p[k] = h[k] / (L \cdot C)$.
5. **CDF:** $\text{cdf}[k] = \sum_{j=0}^{k} p[j]$.
6. **LUT:** $\text{lut}[k] = \text{round}\left(\text{cdf}[k] \cdot (2^B - 1)\right)$.
7. **Aplicação:** $g[i,j] = \text{lut}[f[i,j]]$.
8. **Saída:** Exibir a matriz equalizada $L \times C$.

---

#### 📌 Restrições Computacionais

* **Arredondamento:** Usar arredondamento matemático (`round`) na LUT.
* **Bits:** O número de níveis é $2^B$ (ex.: $B=3 \Rightarrow 8$ níveis, $B=8 \Rightarrow 256$ níveis).
* **CDF acumulada:** $\text{cdf}[k] = \sum_{j=0}^{k} p[j]$, com $\text{cdf}[2^B-1] = 1.0$.

---

#### 🧠 Fundamentação Teórica

| Etapa | Operação | Fórmula |
|:-----:|:---------|:--------|
| 1 | Histograma | $h[k] \leftarrow$ nº pixels com intensidade $k$ |
| 2 | Probabilidade | $p[k] = h[k] / (L \cdot C)$ |
| 3 | CDF | $\text{cdf}[k] = \sum_{j=0}^{k} p[j]$ |
| 4 | LUT | $\text{lut}[k] = \text{round}(\text{cdf}[k] \cdot (2^B-1))$ |
| 5 | Aplicação | $g[i,j] = \text{lut}[f[i,j]]$ |

---

#### 📦 Especificação de Entrada e Saída (VPL)

**Entrada:**
* Linha 1: Inteiro $L$.
* Linha 2: Inteiro $C$.
* Linha 3: Inteiro $B$ (número de bits).
* Linhas seguintes: Elementos inteiros da matriz.

**Saída:**
* Matriz equalizada em $L$ linhas e $C$ colunas.

#### 📌 Exemplos

| Entrada | Saída | Observação |
|---------|-------|------------|
| 5<br>5<br>3<br>3 4 2 3 4<br>4 3 3 4 3<br>2 3 4 3 2<br>3 4 3 2 3<br>4 3 2 3 4 | 5 7 1 5 7<br>7 5 5 7 5<br>1 5 7 5 1<br>5 7 5 1 5<br>7 5 1 5 7 | Exemplo 3 bits do capítulo |
| 1<br>4<br>3<br>0 0 7 7 | 0 0 7 7 | Histograma bimodal extremo |

In [5]:
from IPython.display import HTML
HTML("""
<div id="sim-ep0304" style="background-color:#fef9ef;border-radius:18px;border:1px solid #ede6d8;overflow:hidden;margin-top:20px;font-family:sans-serif;">
  <div style="background:#f3efe6;padding:8px 16px;font-size:12px;color:#5e5a4a;border-bottom:1px solid #e9dfcf;display:flex;justify-content:space-between;align-items:center;">
    <span>🎮 Simulador: Equalização de Histograma (3 bits)</span>
    <span style="background:#e8e0cf;border-radius:40px;padding:2px 10px;font-weight:600;font-size:10px;">📊 lut[k] = round(cdf[k] × 7)</span>
  </div>
  <div style="padding:20px;background:white;">
    <div style="display:grid;grid-template-columns:1fr 1fr;gap:20px;margin-bottom:16px;">
      <div style="text-align:center;background:white;border:1px solid #eee;padding:15px;border-radius:12px;">
        <p style="font-size:10px;font-weight:600;color:#888;text-transform:uppercase;margin-bottom:12px;">Original (3 bits, 0-7)</p>
        <div id="ep0304_grid_orig" style="display:grid;grid-template-columns:repeat(4,42px);gap:4px;justify-content:center;"></div>
        <button id="ep0304_btn_new" style="margin-top:12px;font-size:11px;padding:5px 10px;border-radius:4px;border:1px solid #ccc;background:white;cursor:pointer;">🎲 Nova Imagem</button>
      </div>
      <div style="text-align:center;background:white;border:1px solid #eee;padding:15px;border-radius:12px;">
        <p style="font-size:10px;font-weight:600;color:#888;text-transform:uppercase;margin-bottom:12px;">Equalizada</p>
        <div id="ep0304_grid_eq" style="display:grid;grid-template-columns:repeat(4,42px);gap:4px;justify-content:center;"></div>
      </div>
    </div>
    <div id="ep0304_lut" style="background:#e3f2fd;border-radius:8px;padding:10px;border:1px solid #90caf9;font-family:monospace;font-size:10px;color:#0d47a1;text-align:center;">LUT: calculando...</div>
  </div>
</div>
<script>
setTimeout(function(){
  var gO=document.getElementById('ep0304_grid_orig');
  var gE=document.getElementById('ep0304_grid_eq');
  var lutDiv=document.getElementById('ep0304_lut');
  if(!gO)return;
  var pixels=[];
  var L_bits=3; var L_max=Math.pow(2,L_bits);
  function gen_0304(){pixels=Array.from({length:16},function(){return Math.floor(Math.random()*L_max);});}
  function render_0304(){
    var h=new Array(L_max).fill(0);
    pixels.forEach(function(p){h[p]++;});
    var total=pixels.length;
    var cdf=0; var lut=[];
    for(var k=0;k<L_max;k++){cdf+=h[k]/total;lut.push(Math.round(cdf*(L_max-1)));}
    lutDiv.innerHTML='LUT: ['+lut.join(', ')+']';
    gO.innerHTML=''; gE.innerHTML='';
    pixels.forEach(function(p){
      var sc=Math.round(p/7*255);
      var cO=document.createElement('div');
      cO.style.cssText='width:42px;height:42px;display:flex;align-items:center;justify-content:center;font-size:11px;font-weight:bold;border-radius:4px;border:1px solid #eee;';
      cO.style.background='rgb('+sc+','+sc+','+sc+')';
      cO.style.color=sc>128?'black':'white';
      cO.innerText=p; gO.appendChild(cO);
      var r=lut[p]; var sr=Math.round(r/7*255);
      var cE=document.createElement('div');
      cE.style.cssText='width:42px;height:42px;display:flex;align-items:center;justify-content:center;font-size:11px;font-weight:bold;border-radius:4px;border:1px solid #ddd;';
      cE.style.background='rgb('+sr+','+sr+','+sr+')';
      cE.style.color=sr>128?'black':'white';
      cE.innerText=r; gE.appendChild(cE);
    });
  }
  document.getElementById('ep0304_btn_new').onclick=function(){gen_0304();render_0304();};
  gen_0304(); render_0304();
},100);
</script>
""")

In [ ]:
%%writefile EP03_04.py
# Código Python

In [ ]:
TestSuite("EP03_04.py").run()

---

### EP03_05 🔲 Aplicação de Máscara AND Binária

Em sistemas de inspeção industrial por visão computacional, é necessário isolar regiões de interesse (ROI) em imagens de peças para verificar defeitos de fabricação. A operação **AND bit a bit com uma máscara binária** é o mecanismo fundamental para recortar exatamente a área de inspeção, zerando todos os pixels fora dela.

#### 📋 Diretrizes de Implementação

1. **Dimensões:** Ler os inteiros $L$ (linhas) e $C$ (colunas).
2. **Dados:** Ler a matriz de pixels $f$ (valores $\in [0, 255]$).
3. **Máscara:** Ler a matriz binária $m$ (valores: apenas 0 ou 255).
4. **Mapeamento:** Para cada pixel $(i,j)$, aplicar o AND bit a bit:

$$g(i,j) = f(i,j) \;\text{AND}\; m(i,j)$$

onde $255 = $ `11111111` e $0 = $ `00000000` em binário.

5. **Saída:** Exibir a matriz resultante $L \times C$.

---

#### 📌 Restrições Computacionais

* **AND com 255:** $p \; \text{AND} \; 255 = p$ (todos os bits preservados).
* **AND com 0:** $p \; \text{AND} \; 0 = 0$ (todos os bits zerados).
* **Máscara:** Os únicos valores possíveis na máscara são 0 e 255.
* **Implementação:** Em Python, o AND bit a bit entre inteiros usa o operador `&`.

---

#### 🧠 Fundamentação Teórica

| Pixel $f$ | Máscara $m$ | Resultado $f$ AND $m$ |
|:---------:|:-----------:|:---------------------:|
| qualquer $v$ | 255 (`11111111`) | $v$ (preservado) |
| qualquer $v$ | 0 (`00000000`) | 0 (zerado) |

---

#### 📦 Especificação de Entrada e Saída (VPL)

**Entrada:**
* Linha 1: Inteiro $L$.
* Linha 2: Inteiro $C$.
* Linhas seguintes: Elementos de $f$ ($L$ linhas).
* Linhas seguintes: Elementos de $m$ ($L$ linhas com valores 0 ou 255).

**Saída:**
* Matriz resultante $L \times C$.

#### 📌 Exemplos

| Entrada | Saída | Observação |
|---------|-------|------------|
| 2<br>3<br>100 150 200<br>50 80 120<br>255 255 0<br>0 255 255 | 100 150 0<br>0 80 120 | Máscara seleciona região |
| 1<br>4<br>10 20 30 40<br>255 0 255 0 | 10 0 30 0 | Alternado preservado/zerado |

In [6]:
from IPython.display import HTML
HTML("""
<div id="sim-ep0305" style="background-color:#fef9ef;border-radius:18px;border:1px solid #ede6d8;overflow:hidden;margin-top:20px;font-family:sans-serif;">
  <div style="background:#f3efe6;padding:8px 16px;font-size:12px;color:#5e5a4a;border-bottom:1px solid #e9dfcf;display:flex;justify-content:space-between;align-items:center;">
    <span>🎮 Simulador: Máscara AND Binária</span>
    <span style="background:#e8e0cf;border-radius:40px;padding:2px 10px;font-weight:600;font-size:10px;">🔲 g = f AND m</span>
  </div>
  <div style="padding:20px;background:white;">
    <p style="font-size:11px;color:#666;margin-bottom:12px;">Clique nas células da máscara para alternar entre 0 (preto) e 255 (branco):</p>
    <div style="display:grid;grid-template-columns:1fr 1fr 1fr;gap:12px;">
      <div style="text-align:center;background:white;border:1px solid #eee;padding:12px;border-radius:12px;">
        <p style="font-size:10px;font-weight:600;color:#888;text-transform:uppercase;margin-bottom:8px;">Imagem f</p>
        <div id="ep0305_grid_f" style="display:grid;grid-template-columns:repeat(4,38px);gap:3px;justify-content:center;"></div>
        <button id="ep0305_btn_new" style="margin-top:10px;font-size:11px;padding:4px 8px;border-radius:4px;border:1px solid #ccc;background:white;cursor:pointer;">🎲 Nova</button>
      </div>
      <div style="text-align:center;background:white;border:1px solid #eee;padding:12px;border-radius:12px;">
        <p style="font-size:10px;font-weight:600;color:#888;text-transform:uppercase;margin-bottom:8px;">Máscara m (clique!)</p>
        <div id="ep0305_grid_m" style="display:grid;grid-template-columns:repeat(4,38px);gap:3px;justify-content:center;"></div>
      </div>
      <div style="text-align:center;background:white;border:1px solid #eee;padding:12px;border-radius:12px;">
        <p style="font-size:10px;font-weight:600;color:#888;text-transform:uppercase;margin-bottom:8px;">Resultado g</p>
        <div id="ep0305_grid_g" style="display:grid;grid-template-columns:repeat(4,38px);gap:3px;justify-content:center;"></div>
      </div>
    </div>
    <div id="ep0305_debug" style="margin-top:16px;background:#fff3e0;border-radius:8px;padding:10px;border:1px solid #ffcc80;font-family:monospace;font-size:11px;color:#e65100;text-align:center;">g(i,j) = f(i,j) AND m(i,j) — Clique na máscara para alterar</div>
  </div>
</div>
<script>
setTimeout(function(){
  var gF=document.getElementById('ep0305_grid_f');
  var gM=document.getElementById('ep0305_grid_m');
  var gG=document.getElementById('ep0305_grid_g');
  if(!gF)return;
  var pixels=[]; var mask=new Array(16).fill(255);
  function gen_0305(){pixels=Array.from({length:16},function(){return Math.floor(Math.random()*256);});}
  function render_0305(){
    gF.innerHTML=''; gM.innerHTML=''; gG.innerHTML='';
    for(var i=0;i<16;i++){
      (function(idx){
        var p=pixels[idx]; var m=mask[idx];
        var cF=document.createElement('div');
        cF.style.cssText='width:38px;height:38px;display:flex;align-items:center;justify-content:center;font-size:9px;font-weight:bold;border-radius:4px;border:1px solid #eee;';
        cF.style.background='rgb('+p+','+p+','+p+')'; cF.style.color=p>128?'black':'white'; cF.innerText=p;
        gF.appendChild(cF);
        var cM=document.createElement('div');
        cM.style.cssText='width:38px;height:38px;display:flex;align-items:center;justify-content:center;font-size:9px;font-weight:bold;border-radius:4px;border:2px solid #ff9800;cursor:pointer;';
        cM.style.background=m===255?'white':'black'; cM.style.color=m===255?'black':'white'; cM.innerText=m;
        cM.onclick=function(){mask[idx]=mask[idx]===255?0:255;render_0305();};
        gM.appendChild(cM);
        var r=p&m;
        var cG=document.createElement('div');
        cG.style.cssText='width:38px;height:38px;display:flex;align-items:center;justify-content:center;font-size:9px;font-weight:bold;border-radius:4px;border:1px solid #ddd;';
        cG.style.background='rgb('+r+','+r+','+r+')'; cG.style.color=r>128?'black':'white'; cG.innerText=r;
        gG.appendChild(cG);
      })(i);
    }
  }
  document.getElementById('ep0305_btn_new').onclick=function(){gen_0305();render_0305();};
  gen_0305(); render_0305();
},100);
</script>
""")

In [ ]:
%%writefile EP03_05.py
# Código Python

In [ ]:
TestSuite("EP03_05.py").run()

---

### EP03_06 🌫️ Filtro de Média com Kernel N×N

Em câmeras de veículos autônomos, imagens capturadas em ambientes com chuva ou névoa possuem ruído gaussiano. O **filtro de média** é a primeira linha de defesa para redução de ruído em tempo real, implementado diretamente em hardware nos processadores de sinal de imagem (ISP) dos sensores CMOS.

#### 📋 Diretrizes de Implementação

1. **Dimensões:** Ler os inteiros $L$ (linhas), $C$ (colunas) e $N$ (tamanho do kernel, sempre ímpar).
2. **Dados:** Ler a matriz de pixels $f$.
3. **Filtro de Média:** Para cada pixel $(i,j)$ **interno** (sem bordas), calcular:

$$g(i,j) = \text{round}\left(\frac{1}{N^2} \sum_{s=-(r)}^{r} \sum_{t=-(r)}^{r} f(i+s,\, j+t)\right), \quad r = \lfloor N/2 \rfloor$$

4. **Tratamento de Borda:** Pixels na borda (onde a janela $N \times N$ ultrapassa os limites) devem ser **copiados diretamente** do original sem modificação.
5. **Saída:** Exibir a matriz resultante $L \times C$.

---

#### 📌 Restrições Computacionais

* **Raio:** $r = \lfloor N/2 \rfloor$ (metade do kernel, inteiro).
* **Pixels internos:** $(i,j)$ com $r \le i < L-r$ e $r \le j < C-r$.
* **Arredondamento:** Usar arredondamento matemático antes de converter para inteiro.
* **Sem clipping:** A média de valores $\in [0,255]$ permanece em $[0,255]$.

---

#### 🧠 Fundamentação Teórica

| Tamanho $N$ | Coeficiente | Pixels na janela | Efeito |
|:-----------:|:-----------:|:----------------:|:------:|
| 3 | $1/9 \approx 0.111$ | 9 | Suave |
| 5 | $1/25 = 0.04$ | 25 | Médio |
| 7 | $1/49 \approx 0.020$ | 49 | Forte |

---

#### 📦 Especificação de Entrada e Saída (VPL)

**Entrada:**
* Linha 1: Inteiro $L$.
* Linha 2: Inteiro $C$.
* Linha 3: Inteiro $N$ (ímpar, $N \ge 3$).
* Linhas seguintes: Elementos da matriz original.

**Saída:**
* Matriz filtrada $L \times C$.

#### 📌 Exemplos

| Entrada | Saída | Observação |
|---------|-------|------------|
| 3<br>3<br>3<br>10 20 30<br>40 50 60<br>70 80 90 | 10 20 30<br>40 50 60<br>70 80 90 | Apenas borda (3x3 = borda total) |
| 5<br>5<br>3<br>0 0 0 0 0<br>0 0 0 0 0<br>0 0 100 0 0<br>0 0 0 0 0<br>0 0 0 0 0 | 0 0 0 0 0<br>0 0 0 0 0<br>0 0 11 0 0<br>0 0 0 0 0<br>0 0 0 0 0 | Pixel isolado no centro: round(100/9)=11 |

In [7]:
from IPython.display import HTML
HTML("""
<div id="sim-ep0306" style="background-color:#fef9ef;border-radius:18px;border:1px solid #ede6d8;overflow:hidden;margin-top:20px;font-family:sans-serif;">
  <div style="background:#f3efe6;padding:8px 16px;font-size:12px;color:#5e5a4a;border-bottom:1px solid #e9dfcf;display:flex;justify-content:space-between;align-items:center;">
    <span>🎮 Simulador: Filtro de Média N×N</span>
    <span style="background:#e8e0cf;border-radius:40px;padding:2px 10px;font-weight:600;font-size:10px;">🌫️ g = round(Σ f / N²)</span>
  </div>
  <div style="padding:20px;background:white;">
    <div style="background:#fafafa;border:1px solid #ddd;border-radius:12px;padding:16px;margin-bottom:16px;">
      <div style="display:flex;justify-content:space-between;margin-bottom:8px;">
        <label style="font-size:12px;font-weight:bold;color:#1565c0;">N (tamanho do kernel)</label>
        <span id="ep0306_vl_n" style="font-family:monospace;font-weight:bold;color:#1565c0;">3</span>
      </div>
      <input id="ep0306_sl_n" style="width:100%;accent-color:#1565c0;" max="7" min="3" step="2" type="range" value="3">
    </div>
    <div style="display:grid;grid-template-columns:1fr 1fr;gap:20px;">
      <div style="text-align:center;background:white;border:1px solid #eee;padding:15px;border-radius:12px;">
        <p style="font-size:10px;font-weight:600;color:#888;text-transform:uppercase;margin-bottom:12px;">Original</p>
        <div id="ep0306_grid_orig" style="display:grid;grid-template-columns:repeat(5,36px);gap:3px;justify-content:center;"></div>
        <button id="ep0306_btn_new" style="margin-top:12px;font-size:11px;padding:5px 10px;border-radius:4px;border:1px solid #ccc;background:white;cursor:pointer;">🎲 Nova Imagem</button>
      </div>
      <div style="text-align:center;background:white;border:1px solid #eee;padding:15px;border-radius:12px;">
        <p style="font-size:10px;font-weight:600;color:#888;text-transform:uppercase;margin-bottom:12px;">Filtrada</p>
        <div id="ep0306_grid_filt" style="display:grid;grid-template-columns:repeat(5,36px);gap:3px;justify-content:center;"></div>
      </div>
    </div>
    <div id="ep0306_debug" style="margin-top:16px;background:#e3f2fd;border-radius:8px;padding:10px;border:1px solid #90caf9;font-family:monospace;font-size:11px;color:#0d47a1;text-align:center;">Kernel 3x3: coeficiente = 1/9 ≈ 0.111</div>
  </div>
</div>
<script>
setTimeout(function(){
  var slN=document.getElementById('ep0306_sl_n');
  var vlN=document.getElementById('ep0306_vl_n');
  var gO=document.getElementById('ep0306_grid_orig');
  var gF=document.getElementById('ep0306_grid_filt');
  var db=document.getElementById('ep0306_debug');
  if(!slN||!gO)return;
  var ROWS=5,COLS=5; var pixels=[];
  function gen_0306(){pixels=Array.from({length:ROWS*COLS},function(){return Math.floor(Math.random()*256);});}
  function render_0306(){
    var N=parseInt(slN.value); var r=Math.floor(N/2);
    vlN.innerText=N;
    db.innerHTML='Kernel '+N+'x'+N+': coeficiente = 1/'+N*N+' ≈ '+(1/(N*N)).toFixed(4);
    gO.innerHTML=''; gF.innerHTML='';
    gO.style.gridTemplateColumns='repeat('+COLS+',36px)';
    gF.style.gridTemplateColumns='repeat('+COLS+',36px)';
    var result=pixels.slice();
    for(var i=r;i<ROWS-r;i++){
      for(var j=r;j<COLS-r;j++){
        var sum=0;
        for(var s=-r;s<=r;s++){for(var t=-r;t<=r;t++){sum+=pixels[(i+s)*COLS+(j+t)];}}
        result[i*COLS+j]=Math.round(sum/(N*N));
      }
    }
    for(var i=0;i<ROWS*COLS;i++){
      var p=pixels[i];
      var cO=document.createElement('div');
      cO.style.cssText='width:36px;height:36px;display:flex;align-items:center;justify-content:center;font-size:9px;font-weight:bold;border-radius:3px;border:1px solid #eee;';
      cO.style.background='rgb('+p+','+p+','+p+')'; cO.style.color=p>128?'black':'white'; cO.innerText=p;
      gO.appendChild(cO);
      var q=result[i];
      var cF=document.createElement('div');
      cF.style.cssText='width:36px;height:36px;display:flex;align-items:center;justify-content:center;font-size:9px;font-weight:bold;border-radius:3px;border:1px solid #ddd;';
      cF.style.background='rgb('+q+','+q+','+q+')'; cF.style.color=q>128?'black':'white'; cF.innerText=q;
      gF.appendChild(cF);
    }
  }
  slN.oninput=render_0306;
  document.getElementById('ep0306_btn_new').onclick=function(){gen_0306();render_0306();};
  gen_0306(); render_0306();
},100);
</script>
""")

In [ ]:
%%writefile EP03_06.py
# Código Python

In [ ]:
TestSuite("EP03_06.py").run()

---

### EP03_07 🔍 Operador Laplaciano (w4) para Realce de Bordas

Em tomografias de alta resolução, a nitidez das bordas entre tecidos é crítica para diagnóstico. O **operador Laplaciano** é amplamente utilizado em pipelines de pré-processamento de imagens médicas para realçar automaticamente os contornos anatômicos antes da segmentação, evitando intervenção manual do radiologista.

#### 📋 Diretrizes de Implementação

1. **Dimensões:** Ler os inteiros $L$ (linhas) e $C$ (colunas).
2. **Dados:** Ler a matriz de pixels $f$.
3. **Laplaciano (w4):** Para cada pixel **interno** $(i,j)$ com $1 \le i < L-1$, $1 \le j < C-1$, calcular:

$$\nabla^2 f(i,j) = f(i-1,j) + f(i+1,j) + f(i,j-1) + f(i,j+1) - 4 \cdot f(i,j)$$

4. **Realce:** Calcular a imagem realçada:

$$g(i,j) = \text{clip}(f(i,j) - \nabla^2 f(i,j))$$

5. **Borda:** Pixels na borda são copiados diretamente: $g(i,j) = f(i,j)$.
6. **Saída:** Exibir a matriz realçada $L \times C$.

---

#### 📌 Restrições Computacionais

* **Kernel w4:** $\begin{bmatrix} 0 & 1 & 0 \\ 1 & -4 & 1 \\ 0 & 1 & 0 \end{bmatrix}$ — apenas 4-vizinhos.
* **Saturação:** $\text{clip}(x) = \max(0, \min(255, x))$ aplicado ao resultado do realce.
* **Sem arredondamento:** O Laplaciano usa apenas somas/subtrações de inteiros.

---

#### 🧠 Fundamentação Teórica

| Região | $\nabla^2 f$ | Efeito do Realce |
|:------:|:------------:|:----------------:|
| Uniforme | $\approx 0$ | Sem alteração |
| Borda crescente | $< 0$ | Pixel clareado |
| Borda decrescente | $> 0$ | Pixel escurecido |

---

#### 📦 Especificação de Entrada e Saída (VPL)

**Entrada:**
* Linha 1: Inteiro $L$.
* Linha 2: Inteiro $C$.
* Linhas seguintes: Elementos da matriz original.

**Saída:**
* Matriz realçada $L \times C$.

#### 📌 Exemplos

| Entrada | Saída | Observação |
|---------|-------|------------|
| 3<br>3<br>0 0 0<br>0 100 0<br>0 0 0 | 0 0 0<br>0 500 0<br>0 0 0 | Pico isolado: 100 - (0+0+0+0-400) = 500, clip → 255 espera resultado clipado |
| 3<br>3<br>50 50 50<br>50 50 50<br>50 50 50 | 50 50 50<br>50 50 50<br>50 50 50 | Região uniforme: Laplaciano = 0 |

In [8]:
from IPython.display import HTML
HTML("""
<div id="sim-ep0307" style="background-color:#fef9ef;border-radius:18px;border:1px solid #ede6d8;overflow:hidden;margin-top:20px;font-family:sans-serif;">
  <div style="background:#f3efe6;padding:8px 16px;font-size:12px;color:#5e5a4a;border-bottom:1px solid #e9dfcf;display:flex;justify-content:space-between;align-items:center;">
    <span>🎮 Simulador: Laplaciano w4 — Realce de Bordas</span>
    <span style="background:#e8e0cf;border-radius:40px;padding:2px 10px;font-weight:600;font-size:10px;">🔍 g = clip(f - ∇²f)</span>
  </div>
  <div style="padding:20px;background:white;">
    <div style="display:grid;grid-template-columns:1fr 1fr 1fr;gap:12px;">
      <div style="text-align:center;background:white;border:1px solid #eee;padding:12px;border-radius:12px;">
        <p style="font-size:10px;font-weight:600;color:#888;text-transform:uppercase;margin-bottom:8px;">Original f</p>
        <div id="ep0307_grid_f" style="display:grid;grid-template-columns:repeat(5,36px);gap:3px;justify-content:center;"></div>
        <button id="ep0307_btn_new" style="margin-top:10px;font-size:11px;padding:4px 8px;border-radius:4px;border:1px solid #ccc;background:white;cursor:pointer;">🎲 Nova</button>
      </div>
      <div style="text-align:center;background:white;border:1px solid #eee;padding:12px;border-radius:12px;">
        <p style="font-size:10px;font-weight:600;color:#888;text-transform:uppercase;margin-bottom:8px;">Laplaciano |∇²f|</p>
        <div id="ep0307_grid_lap" style="display:grid;grid-template-columns:repeat(5,36px);gap:3px;justify-content:center;"></div>
      </div>
      <div style="text-align:center;background:white;border:1px solid #eee;padding:12px;border-radius:12px;">
        <p style="font-size:10px;font-weight:600;color:#888;text-transform:uppercase;margin-bottom:8px;">Realçado g</p>
        <div id="ep0307_grid_g" style="display:grid;grid-template-columns:repeat(5,36px);gap:3px;justify-content:center;"></div>
      </div>
    </div>
    <div id="ep0307_debug" style="margin-top:16px;background:#e8eaf6;border-radius:8px;padding:10px;border:1px solid #9fa8da;font-family:monospace;font-size:11px;color:#283593;text-align:center;">Kernel w4: vizinhos N,S,L,O com peso 1; centro com peso -4</div>
  </div>
</div>
<script>
setTimeout(function(){
  var gF=document.getElementById('ep0307_grid_f');
  var gL=document.getElementById('ep0307_grid_lap');
  var gG=document.getElementById('ep0307_grid_g');
  if(!gF)return;
  var ROWS=5,COLS=5; var pixels=[];
  function gen_0307(){pixels=Array.from({length:ROWS*COLS},function(){return Math.floor(Math.random()*256);});}
  function mk_0307(v){
    var c=document.createElement('div');
    c.style.cssText='width:36px;height:36px;display:flex;align-items:center;justify-content:center;font-size:9px;font-weight:bold;border-radius:3px;border:1px solid #eee;';
    c.style.background='rgb('+v+','+v+','+v+')'; c.style.color=v>128?'black':'white'; c.innerText=v; return c;
  }
  function render_0307(){
    gF.innerHTML=''; gL.innerHTML=''; gG.innerHTML='';
    var lap=pixels.slice(); var realc=pixels.slice();
    for(var i=1;i<ROWS-1;i++){
      for(var j=1;j<COLS-1;j++){
        var l=pixels[(i-1)*COLS+j]+pixels[(i+1)*COLS+j]+pixels[i*COLS+(j-1)]+pixels[i*COLS+(j+1)]-4*pixels[i*COLS+j];
        lap[i*COLS+j]=Math.min(255,Math.abs(l));
        realc[i*COLS+j]=Math.max(0,Math.min(255,pixels[i*COLS+j]-l));
      }
    }
    for(var i=0;i<ROWS*COLS;i++){gF.appendChild(mk_0307(pixels[i]));gL.appendChild(mk_0307(lap[i]));gG.appendChild(mk_0307(realc[i]));}
  }
  document.getElementById('ep0307_btn_new').onclick=function(){gen_0307();render_0307();};
  gen_0307(); render_0307();
},100);
</script>
""")

In [ ]:
%%writefile EP03_07.py
# Código Python

In [ ]:
TestSuite("EP03_07.py").run()

---

### EP03_08 🧭 Gradiente de Sobel: Gx e Gy

Em robôs exploradores de Marte (como o Perseverance), a detecção de obstáculos é realizada em tempo real por câmeras estereoscópicas. O **operador de Sobel** calcula o gradiente direcional da cena e é utilizado no algoritmo de detecção de bordas para identificar rochas, fissuras e desníveis do terreno que possam comprometer a navegação.

#### 📋 Diretrizes de Implementação

1. **Dimensões:** Ler os inteiros $L$ (linhas) e $C$ (colunas).
2. **Dados:** Ler a matriz $f$.
3. **Gx e Gy:** Para cada pixel **interno** $(i,j)$ com $1 \le i < L-1$, $1 \le j < C-1$:

$$G_x(i,j) = [f(i-1,j+1) + 2f(i,j+1) + f(i+1,j+1)] - [f(i-1,j-1) + 2f(i,j-1) + f(i+1,j-1)]$$

$$G_y(i,j) = [f(i+1,j-1) + 2f(i+1,j) + f(i+1,j+1)] - [f(i-1,j-1) + 2f(i-1,j) + f(i-1,j+1)]$$

4. **Magnitude:** $|\nabla f(i,j)| = \text{clip}(\text{round}(\sqrt{G_x^2 + G_y^2}))$.
5. **Borda:** Pixels de borda recebem magnitude 0.
6. **Saída:** Exibir a magnitude $L \times C$.

---

#### 📌 Restrições Computacionais

* **Arredondamento:** Aplicar `round` antes de converter para inteiro.
* **Saturação:** $\text{clip}(x) = \max(0, \min(255, x))$.
* **Raiz quadrada:** Usar $\sqrt{G_x^2 + G_y^2}$ (não a aproximação $|G_x| + |G_y|$).

---

#### 🧠 Fundamentação Teórica

| Operador | Detecta | Coeficientes diagonais |
|:--------:|:-------:|:----------------------:|
| $G_x$ | Bordas verticais | $\pm 1$ |
| $G_y$ | Bordas horizontais | $\pm 1$ |
| $|\nabla f|$ | Todas as bordas | Combinado |

---

#### 📦 Especificação de Entrada e Saída (VPL)

**Entrada:**
* Linha 1: Inteiro $L$.
* Linha 2: Inteiro $C$.
* Linhas seguintes: Elementos da matriz.

**Saída:**
* Magnitude do gradiente, matriz $L \times C$.

#### 📌 Exemplos

| Entrada | Saída | Observação |
|---------|-------|------------|
| 3<br>3<br>0 0 0<br>0 0 0<br>0 0 0 | 0 0 0<br>0 0 0<br>0 0 0 | Imagem nula: gradiente zero |
| 3<br>3<br>0 0 255<br>0 0 255<br>0 0 255 | 0 0 0<br>0 255 0<br>0 0 0 | Borda vertical central: Gx alto |

In [9]:
from IPython.display import HTML
HTML("""
<div id="sim-ep0308" style="background-color:#fef9ef;border-radius:18px;border:1px solid #ede6d8;overflow:hidden;margin-top:20px;font-family:sans-serif;">
  <div style="background:#f3efe6;padding:8px 16px;font-size:12px;color:#5e5a4a;border-bottom:1px solid #e9dfcf;display:flex;justify-content:space-between;align-items:center;">
    <span>🎮 Simulador: Gradiente de Sobel</span>
    <span style="background:#e8e0cf;border-radius:40px;padding:2px 10px;font-weight:600;font-size:10px;">🧭 |∇f| = clip(round(√(Gx²+Gy²)))</span>
  </div>
  <div style="padding:20px;background:white;">
    <div style="display:grid;grid-template-columns:1fr 1fr 1fr 1fr;gap:10px;">
      <div style="text-align:center;background:white;border:1px solid #eee;padding:10px;border-radius:12px;">
        <p style="font-size:9px;font-weight:600;color:#888;text-transform:uppercase;margin-bottom:8px;">Original f</p>
        <div id="ep0308_grid_f" style="display:grid;grid-template-columns:repeat(5,30px);gap:2px;justify-content:center;"></div>
        <button id="ep0308_btn_new" style="margin-top:8px;font-size:10px;padding:3px 6px;border-radius:4px;border:1px solid #ccc;background:white;cursor:pointer;">🎲 Nova</button>
      </div>
      <div style="text-align:center;background:white;border:1px solid #eee;padding:10px;border-radius:12px;">
        <p style="font-size:9px;font-weight:600;color:#888;text-transform:uppercase;margin-bottom:8px;">|Gx|</p>
        <div id="ep0308_grid_gx" style="display:grid;grid-template-columns:repeat(5,30px);gap:2px;justify-content:center;"></div>
      </div>
      <div style="text-align:center;background:white;border:1px solid #eee;padding:10px;border-radius:12px;">
        <p style="font-size:9px;font-weight:600;color:#888;text-transform:uppercase;margin-bottom:8px;">|Gy|</p>
        <div id="ep0308_grid_gy" style="display:grid;grid-template-columns:repeat(5,30px);gap:2px;justify-content:center;"></div>
      </div>
      <div style="text-align:center;background:white;border:1px solid #eee;padding:10px;border-radius:12px;">
        <p style="font-size:9px;font-weight:600;color:#888;text-transform:uppercase;margin-bottom:8px;">Magnitude</p>
        <div id="ep0308_grid_mag" style="display:grid;grid-template-columns:repeat(5,30px);gap:2px;justify-content:center;"></div>
      </div>
    </div>
    <div id="ep0308_debug" style="margin-top:16px;background:#fbe9e7;border-radius:8px;padding:10px;border:1px solid #ffab91;font-family:monospace;font-size:11px;color:#bf360c;text-align:center;">Sobel: Gx detecta bordas verticais, Gy detecta bordas horizontais</div>
  </div>
</div>
<script>
setTimeout(function(){
  var gF=document.getElementById('ep0308_grid_f');
  var gGx=document.getElementById('ep0308_grid_gx');
  var gGy=document.getElementById('ep0308_grid_gy');
  var gMag=document.getElementById('ep0308_grid_mag');
  if(!gF)return;
  var ROWS=5,COLS=5; var pixels=[];
  function gen_0308(){pixels=Array.from({length:ROWS*COLS},function(){return Math.floor(Math.random()*256);});}
  function mk_0308(v,w){
    w=w||30;
    var c=document.createElement('div');
    c.style.cssText='width:'+w+'px;height:'+w+'px;display:flex;align-items:center;justify-content:center;font-size:8px;font-weight:bold;border-radius:2px;border:1px solid #eee;';
    c.style.background='rgb('+v+','+v+','+v+')'; c.style.color=v>128?'black':'white'; c.innerText=v; return c;
  }
  function render_0308(){
    gF.innerHTML=''; gGx.innerHTML=''; gGy.innerHTML=''; gMag.innerHTML='';
    var gxV=new Array(ROWS*COLS).fill(0);
    var gyV=new Array(ROWS*COLS).fill(0);
    var magV=new Array(ROWS*COLS).fill(0);
    for(var i=1;i<ROWS-1;i++){
      for(var j=1;j<COLS-1;j++){
        var gx=(pixels[(i-1)*COLS+(j+1)]+2*pixels[i*COLS+(j+1)]+pixels[(i+1)*COLS+(j+1)])-(pixels[(i-1)*COLS+(j-1)]+2*pixels[i*COLS+(j-1)]+pixels[(i+1)*COLS+(j-1)]);
        var gy=(pixels[(i+1)*COLS+(j-1)]+2*pixels[(i+1)*COLS+j]+pixels[(i+1)*COLS+(j+1)])-(pixels[(i-1)*COLS+(j-1)]+2*pixels[(i-1)*COLS+j]+pixels[(i-1)*COLS+(j+1)]);
        gxV[i*COLS+j]=Math.min(255,Math.abs(gx));
        gyV[i*COLS+j]=Math.min(255,Math.abs(gy));
        magV[i*COLS+j]=Math.max(0,Math.min(255,Math.round(Math.sqrt(gx*gx+gy*gy))));
      }
    }
    for(var i=0;i<ROWS*COLS;i++){gF.appendChild(mk_0308(pixels[i]));gGx.appendChild(mk_0308(gxV[i]));gGy.appendChild(mk_0308(gyV[i]));gMag.appendChild(mk_0308(magV[i]));}
  }
  document.getElementById('ep0308_btn_new').onclick=function(){gen_0308();render_0308();};
  gen_0308(); render_0308();
},100);
</script>
""")

In [ ]:
%%writefile EP03_08.py
# Código Python

In [ ]:
TestSuite("EP03_08.py").run()

---

### EP03_09 📡 Filtro da Mediana 3×3

Imagens de radar de abertura sintética (SAR) usadas em monitoramento ambiental e militar sofrem de um tipo específico de ruído chamado *speckle*, que possui características similares ao ruído sal e pimenta. O **filtro da mediana** é o método padrão de remoção desse ruído porque preserva as bordas das estruturas enquanto elimina os pontos espúrios.

#### 📋 Diretrizes de Implementação

1. **Dimensões:** Ler os inteiros $L$ (linhas) e $C$ (colunas).
2. **Dados:** Ler a matriz de pixels $f$.
3. **Filtro da Mediana 3×3:** Para cada pixel **interno** $(i,j)$ com $1 \le i < L-1$, $1 \le j < C-1$:
   - Coletar os 9 pixels da vizinhança $3 \times 3$: $\{f(i+s, j+t) : s,t \in \{-1,0,1\}\}$.
   - Ordenar os 9 valores em ordem crescente.
   - Atribuir $g(i,j)$ ao valor central (posição índice 4, considerando índice 0).

$$g(i,j) = \text{mediana}\{f(i+s, j+t) : s,t \in \{-1,0,1\}\}$$

4. **Borda:** Copiar diretamente: $g(i,j) = f(i,j)$.
5. **Saída:** Exibir a matriz filtrada $L \times C$.

---

#### 📌 Restrições Computacionais

* **Janela:** Sempre $3 \times 3 = 9$ elementos.
* **Mediana:** O elemento central da sequência ordenada (índice 4 de 0 a 8).
* **Sem clipping:** A mediana de valores em $[0, 255]$ permanece em $[0, 255]$.
* **Não linear:** O filtro mediana não pode ser expresso como convolução linear.

---

#### 🧠 Fundamentação Teórica

| Ruído | Filtro de Média | Filtro de Mediana |
|:-----:|:---------------:|:-----------------:|
| Sal e pimenta (0 ou 255) | Espalha o ruído | Remove sem distorcer bordas |
| Gaussiano | Reduz eficazmente | Reduz parcialmente |

---

#### 📦 Especificação de Entrada e Saída (VPL)

**Entrada:**
* Linha 1: Inteiro $L$.
* Linha 2: Inteiro $C$.
* Linhas seguintes: Elementos da matriz.

**Saída:**
* Matriz filtrada $L \times C$.

#### 📌 Exemplos

| Entrada | Saída | Observação |
|---------|-------|------------|
| 3<br>3<br>100 100 100<br>100 0 100<br>100 100 100 | 100 100 100<br>100 100 100<br>100 100 100 | Ponto preto eliminado: mediana de 8×100+1×0 = 100 |
| 3<br>3<br>50 50 50<br>50 255 50<br>50 50 50 | 50 50 50<br>50 50 50<br>50 50 50 | Ponto branco (sal) eliminado |

In [10]:
from IPython.display import HTML
HTML("""
<div id="sim-ep0309" style="background-color:#fef9ef;border-radius:18px;border:1px solid #ede6d8;overflow:hidden;margin-top:20px;font-family:sans-serif;">
  <div style="background:#f3efe6;padding:8px 16px;font-size:12px;color:#5e5a4a;border-bottom:1px solid #e9dfcf;display:flex;justify-content:space-between;align-items:center;">
    <span>🎮 Simulador: Filtro da Mediana 3×3</span>
    <span style="background:#e8e0cf;border-radius:40px;padding:2px 10px;font-weight:600;font-size:10px;">📡 g = mediana(vizinhança 3×3)</span>
  </div>
  <div style="padding:20px;background:white;">
    <div style="background:#fafafa;border:1px solid #ddd;border-radius:12px;padding:14px;margin-bottom:14px;">
      <div style="display:flex;justify-content:space-between;margin-bottom:6px;">
        <label style="font-size:12px;font-weight:bold;color:#c62828;">Densidade de Ruído Sal&Pimenta (%)</label>
        <span id="ep0309_vl_noise" style="font-family:monospace;font-weight:bold;color:#c62828;">20%</span>
      </div>
      <input id="ep0309_sl_noise" style="width:100%;accent-color:#c62828;" max="50" min="0" step="5" type="range" value="20">
    </div>
    <div style="display:grid;grid-template-columns:1fr 1fr 1fr;gap:12px;">
      <div style="text-align:center;background:white;border:1px solid #eee;padding:12px;border-radius:12px;">
        <p style="font-size:10px;font-weight:600;color:#888;text-transform:uppercase;margin-bottom:8px;">Imagem Limpa</p>
        <div id="ep0309_grid_clean" style="display:grid;grid-template-columns:repeat(5,36px);gap:3px;justify-content:center;"></div>
        <button id="ep0309_btn_new" style="margin-top:10px;font-size:11px;padding:4px 8px;border-radius:4px;border:1px solid #ccc;background:white;cursor:pointer;">🎲 Nova</button>
      </div>
      <div style="text-align:center;background:white;border:1px solid #eee;padding:12px;border-radius:12px;">
        <p style="font-size:10px;font-weight:600;color:#888;text-transform:uppercase;margin-bottom:8px;">Com Ruído S&P</p>
        <div id="ep0309_grid_noisy" style="display:grid;grid-template-columns:repeat(5,36px);gap:3px;justify-content:center;"></div>
      </div>
      <div style="text-align:center;background:white;border:1px solid #eee;padding:12px;border-radius:12px;">
        <p style="font-size:10px;font-weight:600;color:#888;text-transform:uppercase;margin-bottom:8px;">Após Mediana</p>
        <div id="ep0309_grid_med" style="display:grid;grid-template-columns:repeat(5,36px);gap:3px;justify-content:center;"></div>
      </div>
    </div>
    <div id="ep0309_debug" style="margin-top:16px;background:#ffebee;border-radius:8px;padding:10px;border:1px solid #ef9a9a;font-family:monospace;font-size:11px;color:#b71c1c;text-align:center;">Mediana remove outliers preservando bordas — ruído 20%</div>
  </div>
</div>
<script>
setTimeout(function(){
  var slN=document.getElementById('ep0309_sl_noise');
  var vlN=document.getElementById('ep0309_vl_noise');
  var gC=document.getElementById('ep0309_grid_clean');
  var gNo=document.getElementById('ep0309_grid_noisy');
  var gMed=document.getElementById('ep0309_grid_med');
  var db=document.getElementById('ep0309_debug');
  if(!slN||!gC)return;
  var ROWS=5,COLS=5; var clean=[];
  function gen_0309(){clean=Array.from({length:ROWS*COLS},function(){return Math.floor(Math.random()*200)+30;});}
  function mk_0309(v){
    var c=document.createElement('div');
    c.style.cssText='width:36px;height:36px;display:flex;align-items:center;justify-content:center;font-size:9px;font-weight:bold;border-radius:3px;border:1px solid #eee;';
    c.style.background='rgb('+v+','+v+','+v+')'; c.style.color=v>128?'black':'white'; c.innerText=v; return c;
  }
  function render_0309(){
    var dens=parseInt(slN.value)/100;
    vlN.innerText=parseInt(slN.value)+'%';
    db.innerHTML='Mediana remove outliers preservando bordas &mdash; ruído '+parseInt(slN.value)+'%';
    var noisy=clean.slice();
    for(var i=0;i<ROWS*COLS;i++){if(Math.random()<dens){noisy[i]=Math.random()<0.5?0:255;}}
    var med=noisy.slice();
    for(var i=1;i<ROWS-1;i++){
      for(var j=1;j<COLS-1;j++){
        var win=[];
        for(var s=-1;s<=1;s++){for(var t=-1;t<=1;t++){win.push(noisy[(i+s)*COLS+(j+t)]);}}
        win.sort(function(a,b){return a-b;});
        med[i*COLS+j]=win[4];
      }
    }
    gC.innerHTML=''; gNo.innerHTML=''; gMed.innerHTML='';
    for(var i=0;i<ROWS*COLS;i++){gC.appendChild(mk_0309(clean[i]));gNo.appendChild(mk_0309(noisy[i]));gMed.appendChild(mk_0309(med[i]));}
  }
  slN.oninput=render_0309;
  document.getElementById('ep0309_btn_new').onclick=function(){gen_0309();render_0309();};
  gen_0309(); render_0309();
},100);
</script>
""")

In [ ]:
%%writefile EP03_09.py
# Código Python

In [ ]:
TestSuite("EP03_09.py").run()

---

### EP03_10 ✨ Unsharp Masking (USM)

Em sistemas de digitalização de documentos históricos e obras de arte, a nitidez das imagens é fundamental para leitura de textos manuscritos e detalhes ornamentais. O **Unsharp Masking (USM)** é o algoritmo de realce de nitidez padrão utilizado em scanners profissionais e softwares como Adobe Photoshop, controlado pelo parâmetro $k$ que determina a intensidade do realce.

#### 📋 Diretrizes de Implementação

1. **Dimensões:** Ler os inteiros $L$ (linhas) e $C$ (colunas).
2. **Parâmetro:** Ler o valor real $k$ (intensidade do realce, $k \ge 0$).
3. **Dados:** Ler a matriz de pixels $f$.
4. **Suavização:** Calcular $\bar{f}$ com filtro de média $3\times3$ (apenas pixels internos; bordas mantidas):

$$\bar{f}(i,j) = \frac{1}{9} \sum_{s=-1}^{1} \sum_{t=-1}^{1} f(i+s, j+t)$$

5. **Máscara de alta frequência:** $m(i,j) = f(i,j) - \bar{f}(i,j)$.
6. **Realce USM:** Para cada pixel interno:

$$g(i,j) = \text{clip}\left(\text{round}\left(f(i,j) + k \cdot m(i,j)\right)\right)$$

7. **Borda:** $g(i,j) = f(i,j)$ (cópia direta).
8. **Saída:** Exibir a matriz realçada $L \times C$.

---

#### 📌 Restrições Computacionais

* **Arredondamento:** Aplicar `round` antes do clipping.
* **Saturação:** $\text{clip}(x) = \max(0, \min(255, x))$.
* **Operações em float:** Calcular $\bar{f}$ e $m$ em ponto flutuante antes de arredondar o resultado final.
* **$k = 0$:** Sem realce — a saída é idêntica à entrada (exceto pelas bordas).

---

#### 🧠 Fundamentação Teórica

| Etapa | Operação | Descrição |
|:-----:|:---------|:----------|
| 1 | $\bar{f} = f * \frac{1}{9}\mathbf{1}_{3\times3}$ | Suavização (baixas frequências) |
| 2 | $m = f - \bar{f}$ | Máscara (altas frequências) |
| 3 | $g = \text{clip}(\text{round}(f + k \cdot m))$ | Realce ponderado |

---

#### 📦 Especificação de Entrada e Saída (VPL)

**Entrada:**
* Linha 1: Inteiro $L$.
* Linha 2: Inteiro $C$.
* Linha 3: Real $k$.
* Linhas seguintes: Elementos da matriz original.

**Saída:**
* Matriz realçada $L \times C$.

#### 📌 Exemplos

| Entrada | Saída | Observação |
|---------|-------|------------|
| 3<br>3<br>0.0<br>100 100 100<br>100 100 100<br>100 100 100 | 100 100 100<br>100 100 100<br>100 100 100 | k=0: sem realce |
| 3<br>3<br>1.0<br>50 50 50<br>50 200 50<br>50 50 50 | 50 50 50<br>50 255 50<br>50 50 50 | k=1: pixel central realçado e saturado |

In [11]:
from IPython.display import HTML
HTML("""
<div id="sim-ep0310" style="background-color:#fef9ef;border-radius:18px;border:1px solid #ede6d8;overflow:hidden;margin-top:20px;font-family:sans-serif;">
  <div style="background:#f3efe6;padding:8px 16px;font-size:12px;color:#5e5a4a;border-bottom:1px solid #e9dfcf;display:flex;justify-content:space-between;align-items:center;">
    <span>🎮 Simulador: Unsharp Masking (USM)</span>
    <span style="background:#e8e0cf;border-radius:40px;padding:2px 10px;font-weight:600;font-size:10px;">✨ g = clip(f + k·(f - f̄))</span>
  </div>
  <div style="padding:20px;background:white;">
    <div style="background:#fafafa;border:1px solid #ddd;border-radius:12px;padding:16px;margin-bottom:16px;">
      <div style="display:flex;justify-content:space-between;margin-bottom:8px;">
        <label style="font-size:12px;font-weight:bold;color:#6a1b9a;">k (Intensidade do Realce)</label>
        <span id="ep0310_vl_k" style="font-family:monospace;font-weight:bold;color:#6a1b9a;">1.0</span>
      </div>
      <input id="ep0310_sl_k" style="width:100%;accent-color:#6a1b9a;" max="4" min="0" step="0.1" type="range" value="1">
    </div>
    <div style="display:grid;grid-template-columns:1fr 1fr 1fr 1fr;gap:10px;">
      <div style="text-align:center;background:white;border:1px solid #eee;padding:10px;border-radius:12px;">
        <p style="font-size:9px;font-weight:600;color:#888;text-transform:uppercase;margin-bottom:8px;">Original f</p>
        <div id="ep0310_grid_f" style="display:grid;grid-template-columns:repeat(5,30px);gap:2px;justify-content:center;"></div>
        <button id="ep0310_btn_new" style="margin-top:8px;font-size:10px;padding:3px 6px;border-radius:4px;border:1px solid #ccc;background:white;cursor:pointer;">🎲 Nova</button>
      </div>
      <div style="text-align:center;background:white;border:1px solid #eee;padding:10px;border-radius:12px;">
        <p style="font-size:9px;font-weight:600;color:#888;text-transform:uppercase;margin-bottom:8px;">Suavizado f̄</p>
        <div id="ep0310_grid_smooth" style="display:grid;grid-template-columns:repeat(5,30px);gap:2px;justify-content:center;"></div>
      </div>
      <div style="text-align:center;background:white;border:1px solid #eee;padding:10px;border-radius:12px;">
        <p style="font-size:9px;font-weight:600;color:#888;text-transform:uppercase;margin-bottom:8px;">Máscara m+128</p>
        <div id="ep0310_grid_mask" style="display:grid;grid-template-columns:repeat(5,30px);gap:2px;justify-content:center;"></div>
      </div>
      <div style="text-align:center;background:white;border:1px solid #eee;padding:10px;border-radius:12px;">
        <p style="font-size:9px;font-weight:600;color:#888;text-transform:uppercase;margin-bottom:8px;">Realçado g</p>
        <div id="ep0310_grid_g" style="display:grid;grid-template-columns:repeat(5,30px);gap:2px;justify-content:center;"></div>
      </div>
    </div>
    <div id="ep0310_debug" style="margin-top:16px;background:#ede7f6;border-radius:8px;padding:10px;border:1px solid #b39ddb;font-family:monospace;font-size:11px;color:#4527a0;text-align:center;">Fórmula: clip(round(f + 1.0 × (f - f̄)))</div>
  </div>
</div>
<script>
setTimeout(function(){
  var slK=document.getElementById('ep0310_sl_k');
  var vlK=document.getElementById('ep0310_vl_k');
  var gF=document.getElementById('ep0310_grid_f');
  var gS=document.getElementById('ep0310_grid_smooth');
  var gMask=document.getElementById('ep0310_grid_mask');
  var gG=document.getElementById('ep0310_grid_g');
  var db=document.getElementById('ep0310_debug');
  if(!slK||!gF)return;
  var ROWS=5,COLS=5; var pixels=[];
  function gen_0310(){pixels=Array.from({length:ROWS*COLS},function(){return Math.floor(Math.random()*256);});}
  function mk_0310(v){
    var c=document.createElement('div');
    c.style.cssText='width:30px;height:30px;display:flex;align-items:center;justify-content:center;font-size:8px;font-weight:bold;border-radius:2px;border:1px solid #eee;';
    c.style.background='rgb('+v+','+v+','+v+')'; c.style.color=v>128?'black':'white'; c.innerText=v; return c;
  }
  function render_0310(){
    var k=parseFloat(slK.value);
    vlK.innerText=k.toFixed(1);
    db.innerHTML='Fórmula: <b>clip(round(f + '+k.toFixed(1)+' × (f - f̄)))</b>';
    var smooth=pixels.slice();
    for(var i=1;i<ROWS-1;i++){
      for(var j=1;j<COLS-1;j++){
        var sum=0;
        for(var s=-1;s<=1;s++){for(var t=-1;t<=1;t++){sum+=pixels[(i+s)*COLS+(j+t)];}}
        smooth[i*COLS+j]=sum/9;
      }
    }
    gF.innerHTML=''; gS.innerHTML=''; gMask.innerHTML=''; gG.innerHTML='';
    for(var i=0;i<ROWS*COLS;i++){
      var p=pixels[i]; var s=Math.round(smooth[i]);
      var mask=Math.max(0,Math.min(255,Math.round(p-smooth[i])+128));
      var g=Math.max(0,Math.min(255,Math.round(p+k*(p-smooth[i]))));
      gF.appendChild(mk_0310(p)); gS.appendChild(mk_0310(s)); gMask.appendChild(mk_0310(mask)); gG.appendChild(mk_0310(g));
    }
  }
  slK.oninput=render_0310;
  document.getElementById('ep0310_btn_new').onclick=function(){gen_0310();render_0310();};
  gen_0310(); render_0310();
},100);
</script>
""")

In [ ]:
%%writefile EP03_10.py
# Código Python

In [ ]:
TestSuite("EP03_10.py").run()